# College selectivity project: starter notebook

**The question behind the project:** what does it mean for a college to be *selective*, and how does selectivity relate to the **family incomes** of the students who attend, what they pay, and where they end up?

This notebook does **not** do the analysis for you. It shows you where the data lives, what's in it, and where it can mislead you. Read the strengths and weaknesses sections before you decide what to build.

## 1. Load the data

One row per U.S. college, combining three public sources (details below). The file lives in the class GitHub repo.

In [ ]:
import pandas as pd

URL = "https://raw.githubusercontent.com/sawula/ds-course-data/main/college_dataset.csv"
df = pd.read_csv(URL)

print(df.shape)
df.head()

## 2. What's in here

About 145 columns, in five families:

| Family | Example columns | Where it comes from |
|---|---|---|
| **Who** | `inst_name`, `state_abbr`, `inst_control` | IPEDS directory |
| **Selectivity** | `applicants`, `admitted`, `enrolled`, `admit_rate`, `yield` | IPEDS admissions |
| **Test scores** | `sat_total_25`, `sat_total_75`, `act_composite_25`, `act_composite_75`, `sat_percent_submitting` | IPEDS admissions |
| **What families pay, by income** | `net_price_inc1` ... `net_price_inc5`, `pell_pct` | IPEDS financial aid |
| **Parent income and outcomes** | everything starting with `oi_`, e.g. `oi_par_median`, `oi_par_top1pc`, `oi_mr_kq5_pq1` | Opportunity Insights (Chetty et al.) |

The suffixes `_inc1` to `_inc5` are **family income brackets**: 1 = $0-30k, 2 = $30-48k, 3 = $48-75k, 4 = $75-110k, 5 = $110k+, and `_inc99` = all students. (`inst_control`: 1 = public, 2 = private nonprofit, 3 = private for-profit.) Part of your job is to confirm what the codes mean. The full column-by-column description is in [`college_dataset_README.md`](https://github.com/sawula/ds-course-data/blob/main/college_dataset_README.md), and the Opportunity Insights columns are documented in their codebook.

In [ ]:
# How many columns are in each family?
oi_cols  = [c for c in df.columns if c.startswith("oi_")]
inc_cols = [c for c in df.columns if "_inc" in c]
print("Opportunity Insights columns:", len(oi_cols))
print("Income-bracket columns:", len(inc_cols))
print("Everything else:", df.shape[1] - len(oi_cols) - len(inc_cols))

## 3. Strengths of this dataset

- **Broad.** 5,346 colleges, from Harvard to community colleges and beauty schools.
- **Authoritative sources.** Admissions and aid come from IPEDS, the federal government's college data survey (retrieved through the Urban Institute's Education Data Portal). Parent income and mobility come from Raj Chetty's team, who used *tax records* for millions of students rather than surveys.
- **Income shows up three different ways**, so you can check one measure against another: what families at each income level *pay* (net price), what share of students get *Pell grants*, and the *actual parent incomes* of students (Chetty).
- **Outcomes, not just inputs.** Chetty's columns include students' own incomes later in life and the chance that a student from the bottom fifth of incomes reaches the top fifth.
- **Reproducible.** The notebook that built this file is in the repo (`college_dataset_builder.ipynb`). You can see every step, and rerun it.

## 4. Weaknesses: read these before you trust a chart

- **Lots of missing data, and it's not random.** Only about 1,850 colleges have an admit rate, about 960 have SAT/ACT scores, and about 2,015 have Chetty data. Only about 1,153 have *both* an admit rate and Chetty data. Every analysis you do silently drops the rest. Find out who gets dropped.
- **Different years.** The IPEDS columns are for 2021. Chetty's data describes students who attended college roughly 1999-2013. Don't treat the file as one snapshot in time.
- **Different dollars.** Check the documentation for which year's dollars each income column uses (Chetty reports in inflation-adjusted 2015 dollars, as far as I know, and the IPEDS numbers are 2021 dollars). Comparing them directly is a trap.
- **Percentiles, not averages.** IPEDS publishes the 25th and 75th percentile of SAT/ACT scores, not the average. The median (`_50`) columns are empty in this file. Also, `sat_total_*` is the *sum* of two section percentiles, which is an approximation.
- **Test-optional policies.** Many colleges stopped requiring tests around 2020. Look at `sat_percent_submitting` before comparing scores across schools.
- **Net price is not for everyone.** The net-price-by-income numbers describe first-time, full-time students who receive federal aid, so they aren't a typical-family average.
- **Mixed scales.** `pell_pct` is a fraction (0.19), but `sat_percent_submitting` is 0-100 (19). Check before you plot them together.
- **Some "colleges" aren't what you picture.** Cosmetology schools, seminaries, and tiny programs are in here. An `admit_rate` of exactly 1.0 usually means open admissions, and an `admit_rate` of 0 is a tiny school that admitted nobody.
- **The Chetty link is imperfect.** Colleges were matched by ID, and then by name. `oi_match` and `oi_name_similarity` tell you how, and a few matches are probably wrong. Check suspicious rows by hand.
- **Small numbers are noisy.** Chetty's mobility rate for a very selective college is based on very few students from the bottom of the income distribution.
- **Correlation is not causation.** A college can *look* like it moves students up simply because of who it admits.

In [ ]:
# How much of the data is actually there? (non-missing counts out of the total)
key = ["admit_rate", "yield", "sat_total_25", "act_composite_25", "sat_percent_submitting",
       "net_price_inc1", "net_price_inc5", "pell_pct", "oi_par_median", "oi_mr_kq5_pq1"]
df[key].notna().sum()

## 5. Before you write analysis code

Answer these in your notes (or on your spec card) *first*:

1. **What exactly is your question?** (One sentence. Which columns would answer it?)
2. **How will you measure "selective"?** Admit rate? Test scores? Yield? What does each one miss?
3. **How will you measure a student's family income?** Net price by bracket, `pell_pct`, or Chetty's parent income? Why that one?
4. **Which colleges stay in your analysis?** Say how you decide, and how many are left.
5. **What would surprise you,** and what would you check if it happened?

## 6. Questions to get you thinking (pick one or invent your own)

- Do more selective colleges enroll students from richer families? Does the answer depend on *how* you measure selectivity?
- Is a very selective college affordable to a low-income family? How does net price for the lowest and highest income brackets compare across colleges?
- Which colleges are unusual: selective *and* low-income-friendly, or not selective but expensive?
- Does the picture differ between public and private colleges?
- Is there a college whose numbers don't agree across sources? Why might that be?

## 7. Your work

In [ ]:
# Start here.

---
## Sources and credit

- **IPEDS** (National Center for Education Statistics), retrieved through the [Urban Institute Education Data Portal](https://educationdata.urban.org). Please follow the portal's terms of use.
- **Opportunity Insights**, Mobility Report Cards: Chetty, Friedman, Saez, Turner, and Yagan, "Mobility Report Cards: The Role of Colleges in Intergenerational Mobility." Data from [opportunityinsights.org/data](https://opportunityinsights.org/data/).
- Data dictionary and caveats: [`college_dataset_README.md`](https://github.com/sawula/ds-course-data/blob/main/college_dataset_README.md). Builder notebook: `college_dataset_builder.ipynb`.